In [1]:
import torch
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

In [2]:
from transformers import BertModel, BertTokenizerFast

def tokenize_and_align_labels(examples, tokenizer: BertTokenizerFast):
    """
    Tokenize the input samples using the given tokenizer and
    align the target labels with the tokens produced by the tokenizer.

    Parameters:
    - samples (): Input tensor of shape [batch_size, seq_len, embed_size].
    - tokenizer (BertTokenizerFast): tokenizer used to tokenize the data

    Returns:
    - DatasetDict: Output the dataset containing the aligned target labels.

    Hint:
    - Remember that BERT's tokenizer may split a single word into multiple subword tokens.
      This means the number of tokens != the number of original words, so labels must be re-aligned.
    - Use tokenized_inputs.word_ids(batch_index=i) to get, for each token, the index of the
      original word it came from. Special tokens ([CLS], [SEP], [PAD]) will return None.
    - Only assign the real label to the FIRST subword token of each word.
      All other subword tokens (and special tokens) should get label -100,
      which tells PyTorch to ignore them in the loss computation.
    """
    ## INSERT YOUR CODE HERE ##
    # Tokenize the tokens in the dataset
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = [] # list of new labels
    for i, label in enumerate(examples[f"ner_tags"]):
        # Map each token in the tokenized input to their respective word in the original sample words,
        # i.e. you obtain a list containing for each output token the index of the associated original word.
        # If the index is None, it means a special token has been added to the input.
        # Hint: word_ids() returns a list where each position corresponds to a token,
        # and the value is the index of the original word that token came from.
        word_ids = tokenized_inputs.word_ids(batch_index=i)   # Map tokens to their respective word.

        previous_word_idx = None  # index of the last checked word
        label_ids = []
        for word_idx in word_ids:
            # Set the special tokens to -100 to be ignored in loss computation.
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # This is the FIRST token of a new word.
                # Assign the real label for this word.
                label_ids.append(label[word_idx])
            else:
                # This is a continuation subword token (same word as previous token).
                # We do NOT want to double-count this word's label, so assign -100.
                label_ids.append(-100)
            # Updated the id of the last checked word
            previous_word_idx = word_idx
        labels.append(label_ids)

    # Add the aligned labels back into the tokenized_inputs dictionary in the 'labels' field
    tokenized_inputs["labels"] = labels
    ## END OF YOUR CODE ##
    return tokenized_inputs

/Users/robertsparks/Documents/neural_language_modelling/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from datasets import load_dataset

wnut = load_dataset("wnut_17", trust_remote_code=True)  # Load the dataset from HuggingFace

pretrained_model_name = 'bert-base-uncased'
tokenizer = BertTokenizerFast.from_pretrained(pretrained_model_name)

tokenized_wnut = wnut.map(tokenize_and_align_labels, batched=True, fn_kwargs={'tokenizer': tokenizer})

train_dataset_hf, eval_dataset_hf, test_dataset_hf = tokenized_wnut["train"], tokenized_wnut["validation"], tokenized_wnut["test"]

In [4]:
label_list = wnut["train"].features[f"ner_tags"].feature.names
print('Available NER tags:', label_list)

Available NER tags: ['O', 'B-corporation', 'I-corporation', 'B-creative-work', 'I-creative-work', 'B-group', 'I-group', 'B-location', 'I-location', 'B-person', 'I-person', 'B-product', 'I-product']


In [5]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

### Initiliase the Model

In [6]:
import torch.nn as nn
from torch.nn import CrossEntropyLoss

class BertClassifier(nn.Module):
    def __init__(self, bert_model: BertModel, num_labels: int):
        super(BertClassifier, self).__init__()
        """
          Parameters:
            - bert_model (BertModel): Pretrained BERT model to initialise our classifier.
            - num_labels (int): The number of labels for our classifier's output.

          Attributes:
            - bert (BertModel): BERT model.
            - num_labels (int): Number of labels in output.
            - classifier (nn.Linear): Linear transformation for the output of the
                pretrained model to get the labels prediction.
            - dropout (nn.Dropout): Dropout layer to prevent overfitting.
        """

        # use the model given in input to the init function, which is a pretrained BERT model
        self.bert = bert_model

        # Number of labels in the sequence classification task
        self.num_labels = num_labels

        # We can access the information like hidden size, vocab, #layers, ect..
        # of a pretrained model in its config structure, which is accessible at model.config
        # https://huggingface.co/docs/transformers/model_doc/bert#transformers.BertConfig

        # Classifier layer to map the output of BERT to label space
        self.classifier = nn.Linear(bert_model.config.hidden_size, num_labels)

        # Dropout layer to prevent overfitting
        # here we used the same dropout value as the model hidden states dropout
        self.dropout = nn.Dropout(bert_model.config.hidden_dropout_prob)


    def forward(self, input_ids: torch.Tensor,
                attention_mask: torch.Tensor,
                token_type_ids: torch.Tensor,
                labels: torch.Tensor=None):
        """
        Forward pass for the BERT-based classifier.

        Parameters:
        - input_ids (torch.Tensor): tensor containing the ids of the tokenised input.
        - attention_mask (torch.Tensor): tensor containing the attention mask of the tokenised input.
        - token_type_ids (torch.Tensor): tensor containing the token type ids of the tokenised input.
        - labels (torch.Tensor): tensor containing the aligned target labels.

        Returns:
        - torch.Tensor: Output of the classifier.
        """

        # Get the outputs from pretrained BERT
        outputs = self.bert(input_ids=input_ids,
                            attention_mask=attention_mask,
                            token_type_ids=token_type_ids)

        # we will use the last hidden state information, since it's an accumulation of processed information in previous layers
        # Some practitioner, instead of the last hidden state, they take all layers hidden states and average them.
        # For many tasks last hidden state works the best.
        # BERT model also return all hidden states and you can access it using outputs.hidden_states
        sequence_output = outputs.last_hidden_state
        sequence_output = self.dropout(sequence_output)
        logits = self.classifier(sequence_output)

        loss = None
        if labels is not None:
            loss_fct = CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))

        output = (logits,) + outputs[2:]
        return ((loss,) + output) if loss is not None else output


In [7]:
# Let's initialise our classification model
bert_model = BertModel.from_pretrained(pretrained_model_name) # Load a pretrained BERT model
num_labels = len(wnut["train"].features[f"ner_tags"].feature.names)  # Update this based on your NER task

model = BertClassifier(bert_model, num_labels=num_labels).to(device)  # init our classifier model

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9504.29it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Training Arguments

In [8]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./model_output',    # Directory where model checkpoints and outputs will be saved.
    num_train_epochs=3,# Total number of training epochs.
    per_device_train_batch_size=16, # Batch size per device during training.
    per_device_eval_batch_size=64,  # Batch size for evaluation.
    learning_rate=5e-5,             # Learning rate
    warmup_steps=500,               # Number of warmup steps for learning rate scheduler.
    weight_decay=0.01,              # Weight decay if we apply some.
    logging_dir='./logs',           # Directory for storing logs.
    logging_steps=10,               # Log every X updates steps.
    eval_strategy="steps",    # Evaluate every X steps.
    eval_steps=50,                  # Number of steps to evaluate after.
    save_strategy="steps",          # The checkpoint save strategy to use.
    save_steps=100,                 # Save checkpoint every X steps.
    load_best_model_at_end=True,     # Whether to load the best model found at each evaluation.
    report_to="none"                 # The list of integrations to report the results and logs to.
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


## Compute Metrics

In [9]:
import evaluate
import numpy as np

metric = evaluate.load("seqeval")

def compute_metrics(model_output):
    """
    Compute evaluation metrics to check the performance of the model during training (on the validation set)
    and after training (on the test set).
    The input parameter of the function is a Tuple as required by the HuggingFace Trainer.

    Parameters:
    - model_output (Tuple): Contains model's raw predictions and target labels.

    Returns:
    - dict: Dictionary of the evaluation metrics, i.e. Precision, Recall, F1, and Accuracy.

    Returns:
    - dict: Dictionary of the evaluation metrics, i.e. Precision, Recall, F1, and Accuracy.

    Hint:
    - model_output is a tuple of (predictions, labels). predictions contains raw logits of shape
      [num_samples, seq_len, num_labels],  one score per label per token position.
    - To get the predicted class for each token, you need to pick the index of the highest logit.
      Think about which axis to apply np.argmax on.
    - Both predictions and labels still contain positions for special tokens ([CLS], [SEP], [PAD]).
      These were assigned label -100 during preprocessing, and must be filtered out before
      computing metrics. seqeval expects string label names (e.g. "B-person"), not integers.
    - label_list is available in the notebook scope and maps integer indices to string label names.
    """
    ## INSERT YOUR CODE HERE ##
    # Unpack model_output into predictions (raw logits) and labels.
    predictions, labels = model_output

    # Convert logits to predicted class indices by selecting the index of the max logit in each token position.
    predictions = np.argmax(predictions, axis=2)

    # Filter out the special tokens and align the predictions with the actual labels.
    # This step is necessary because models like BERT use special tokens (e.g., [CLS], [SEP], [PAD]),
    # and labels for these tokens are typically set to an ignore index (e.g., -100) so they don't affect loss computation.
    true_predictions = [
    [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    # Compute evaluation metrics such as precision, recall, F1 score, and accuracy using the 'metric' object.
    # This 'metric' object is typically an instance from the Hugging Face's `datasets` library, which provides
    # various metrics calculation functions.
    # Compute the metrics using the preloaded seqeval metric object.
    results = metric.compute(predictions=true_predictions, references=true_labels)

    # Return a dictionary containing the computed metrics.
    return {
    "precision": results["overall_precision"],
    "recall": results["overall_recall"],
    "f1": results["overall_f1"],
    "accuracy": results["overall_accuracy"],
}
  ## END OF YOUR CODE ##


In [10]:
from transformers import Trainer

trainer = Trainer(
    model=model,                         # The model to be trained or fine-tuned.
    args=training_args,                  # TrainingArguments object containing the training and evaluation configurations.
    train_dataset=train_dataset_hf,         # The dataset to be used during training. Should be a Hugging Face Dataset or a dataset that implements __len__ and __getitem__.
    eval_dataset=eval_dataset_hf,           # The dataset for evaluation. Similar format as train_dataset. Used to evaluate the model performance at each logging step or epoch end.
    compute_metrics=compute_metrics,     # A function that computes metrics of interest for evaluation. It takes an EvalPrediction object (which has .predictions and .label_ids attributes) and should return a dictionary mapping metric names to their values.
    data_collator=data_collator          # Ensures samples are properly padded and batched
    )


## Training the model

In [11]:
train_output = trainer.train()

/Users/robertsparks/Documents/neural_language_modelling/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
50,1.042397,0.929639,0.000000,0.000000,0.000000,0.920465
100,0.312752,0.414849,0.000000,0.000000,0.000000,0.920529
150,0.177878,0.298393,0.462329,0.161483,0.239362,0.930638
200,0.177565,0.253770,0.525562,0.307416,0.387925,0.938521
250,0.171803,0.286291,0.645963,0.373206,0.473086,0.943544
300,0.129456,0.260478,0.722343,0.398325,0.513493,0.944879
350,0.155800,0.197054,0.654854,0.508373,0.572391,0.953589
400,0.093682,0.233377,0.676678,0.458134,0.546362,0.951999
450,0.089091,0.243753,0.663755,0.545455,0.598818,0.955433
500,0.078157,0.211106,0.579384,0.584928,0.582143,0.952826


/Users/robertsparks/Documents/neural_language_modelling/.venv/lib/python3.12/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/robertsparks/Documents/neural_language_modelling/.venv/lib/python3.12/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/robertsparks/Documents/neural_language_modelling/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/robertsparks/Documents/neural_langu

## Evaluating the model

In [13]:
# evaluate the best trained model on the validation set
print('Evaluation metrics on the validation set:')
trainer.evaluate()

Evaluation metrics on the validation set:


RuntimeError: on_train_begin must be called before on_evaluate

## Making predictions

In [ ]:
predictions = trainer.predict(test_dataset_hf)
print('Evaluation metrics on the test set:')
# predict() return multiple information, such as predicted logits for each sample and
# evaluation metrics computed using the compute_metrics function.
# In this moment, we're interested in accessing the evaluation metric to evaluate how good our network is
predictions.metrics